# 06 - Visualización de Resultados

Este notebook genera todas las visualizaciones finales para la memoria del TFG.

## Contenido
1. Gráficos de métricas
2. Análisis de errores
3. Importancia de características
4. Figuras para la memoria

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils.visualization import Visualizer

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

%matplotlib inline

In [ ]:
# Crear directorio de figuras si no existe
figures_dir = Path('../reports/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

viz = Visualizer(output_dir=str(figures_dir))

## 1. Cargar Resultados

In [ ]:
# Intentar cargar resultados guardados
results_path = Path('../reports/results/final_model_comparison.csv')

if results_path.exists():
    comparison_df = pd.read_csv(results_path)
    print("Resultados cargados desde archivo")
else:
    # Crear datos de ejemplo para demostración
    comparison_df = pd.DataFrame({
        'model': ['naive_bayes', 'logistic_regression', 'random_forest', 'xgboost'],
        'accuracy': [0.92, 0.95, 0.97, 0.98],
        'precision': [0.90, 0.94, 0.96, 0.97],
        'recall': [0.88, 0.93, 0.95, 0.96],
        'f1': [0.89, 0.935, 0.955, 0.965],
        'roc_auc': [0.94, 0.97, 0.98, 0.99]
    })
    print("Usando datos de ejemplo")

display(comparison_df)

## 2. Gráficos de Métricas

In [ ]:
# Gráfico de barras - Todas las métricas
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['accuracy', 'precision', 'recall', 'f1']
x = np.arange(len(comparison_df))
width = 0.2

colors = ['#2E86AB', '#A23B72', '#28A745', '#FFC107']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*width, comparison_df[metric], width, label=metric.capitalize(), color=color)
    
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparación de Métricas por Modelo', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison_df['model'], rotation=45, ha='right')
ax.legend(loc='lower right')
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / 'metrics_comparison_final.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico de radar
from math import pi

categories = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = plt.cm.Set2(np.linspace(0, 1, len(comparison_df)))

for idx, row in comparison_df.iterrows():
    values = [row['accuracy'], row['precision'], row['recall'], row['f1'], row['roc_auc']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['model'], color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim([0.85, 1.0])
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.set_title('Comparación de Modelos (Radar)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'radar_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Gráfico de F1 Score

In [ ]:
# F1 Score por modelo (horizontal)
fig, ax = plt.subplots(figsize=(10, 5))

sorted_df = comparison_df.sort_values('f1', ascending=True)
colors = plt.cm.RdYlGn(sorted_df['f1'])

bars = ax.barh(sorted_df['model'], sorted_df['f1'], color=colors)

# Añadir valores
for bar, value in zip(bars, sorted_df['f1']):
    ax.text(value + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{value:.4f}', va='center', fontsize=11)

ax.set_xlabel('F1 Score', fontsize=12)
ax.set_title('F1 Score por Modelo', fontsize=14, fontweight='bold')
ax.set_xlim([0.85, 1.05])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(figures_dir / 'f1_scores.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Tabla Resumen para LaTeX

In [ ]:
# Generar tabla LaTeX
latex_table = comparison_df.to_latex(
    index=False,
    float_format='%.4f',
    caption='Comparación de rendimiento de modelos',
    label='tab:model_comparison'
)

print("Tabla LaTeX para la memoria:")
print(latex_table)

# Guardar tabla
with open(figures_dir / 'model_comparison_table.tex', 'w') as f:
    f.write(latex_table)

## 5. Resumen Final

In [ ]:
# Mejor modelo
best_model = comparison_df.loc[comparison_df['f1'].idxmax()]

print("="*60)
print("RESUMEN FINAL DE RESULTADOS")
print("="*60)
print(f"\nMejor modelo: {best_model['model']}")
print(f"\nMétricas del mejor modelo:")
print(f"  - Accuracy:  {best_model['accuracy']:.4f}")
print(f"  - Precision: {best_model['precision']:.4f}")
print(f"  - Recall:    {best_model['recall']:.4f}")
print(f"  - F1 Score:  {best_model['f1']:.4f}")
print(f"  - ROC-AUC:   {best_model['roc_auc']:.4f}")
print("="*60)

In [ ]:
# Lista de figuras generadas
print("\nFiguras generadas:")
for fig_file in figures_dir.glob('*.png'):
    print(f"  - {fig_file.name}")

## Figuras para incluir en la memoria:

1. **metrics_comparison_final.png** - Comparación de métricas por modelo
2. **radar_comparison.png** - Gráfico de radar con todas las métricas
3. **f1_scores.png** - F1 Score ordenado por modelo
4. **confusion_matrices_all.png** - Matrices de confusión
5. **roc_curves_comparison.png** - Curvas ROC comparativas

Todas las figuras están guardadas en `reports/figures/` con alta resolución (300 DPI).